## Задание № 3

In [2]:
import pandas as pd
import numpy as np

In [35]:
# Считываемые данные из файла csv
df = pd.read_csv('Data/Electronic_sales_Sep2023-Sep2024.csv')

In [36]:
# Проанализируемые датасет
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  object 
 3   Loyalty Member     20000 non-null  object 
 4   Product Type       20000 non-null  object 
 5   SKU                20000 non-null  object 
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  object 
 8   Payment Method     20000 non-null  object 
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  object 
 13  Shipping Type      20000 non-null  object 
 14  Add-ons Purchased  15132 non-null  object 
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4

In [ ]:
# Выведем первые пять строк
df.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [31]:
# Посмотрим есть ли еще в столбе 'Order Status' незавершенные или отмененные заказы
df['Order Status'].unique()

array(['Cancelled', 'Completed'], dtype=object)

In [32]:
# Такие заказы есть, поэтому отфильтруем датафрейм по завершенным заказам
completed_orders = df[df['Order Status'] == 'Completed']

In [ ]:
# Определим предпочитаемый метод оплаты для каждого покупателя
# Сначала сгруппируем по id и методу оплаты, затем применим агреггирующую функцию, в которую передадим лямбда-функцию,
# фильтрующую по наиболее частому методу (при отсутствии данных выставляем 'Unknown'),
# переиндексируем полученные данные, переименовыаем колонки 
preferred_payment = completed_orders.groupby('Customer ID')['Payment Method']\
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')\
    .reset_index()\
    .rename(columns={'Payment Method': 'Preferred Payment Method'})
preferred_payment.head() # для первых пяти покупателей    

,Customer ID,Preferred Payment Method
0,1000,Paypal
1,1002,Cash
2,1003,Cash
3,1004,Credit Card
4,1005,Debit Card


In [ ]:
# Определим общие траты по каждому покупателю
# Сначала сгруппируем по id и общей сумме транзакции,
# переиндексируем, переименовываем колонки
total_spending = completed_orders.groupby('Customer ID')['Total Price']\
    .sum()\
    .reset_index()\
    .rename(columns={'Total Price': 'Total Spending'}).sort_values('Total Spending', ascending=False) # отсотртируем по наибольшей трате
total_spending    

,Customer ID,Total Spending
5048,11476,29937.93
7095,15399,29084.88
6177,13635,28093.28
7584,16357,27486.01
4960,11332,26725.65
...,...,...
601,2108,20.75
3637,7900,20.75
2447,5587,20.75
2398,5507,20.75


In [44]:
# Определим траты на дополнительные услуги и аксессуары
# Группируем по ID и тратам на аксессуары
addon_spending = completed_orders.groupby('Customer ID')['Add-on Total']\
    .sum()\
    .reset_index()\
    .rename(columns={'Add-on Total': 'Add-on Spending'}).sort_values('Add-on Spending', ascending=False) # отсотртируем по наибольшей трате
addon_spending    

,Customer ID,Add-on Spending
7937,17054,658.32
7221,15648,640.44
5139,11669,613.83
5749,12780,595.63
4926,11270,580.85
...,...,...
6918,15088,0.00
94,1180,0.00
98,1187,0.00
100,1189,0.00
